# Whisper ASR cho corpus AIC 2026 — chạy trên Kaggle

Sinh phụ đề tiếng Việt cho **873 video / 130,7 giờ** để làm kênh văn bản cho Elasticsearch.

**Vì sao cần:** 2/3 corpus là L26 nấu ăn (43,8h) + L25 bài giảng (36,2h) — chỗ CLIP thuần bó tay
(đo được L26 3/8, L25 4/7 lọt top-100) nhưng người ta **nói thành lời**: giảng viên đọc slide,
người dẫn đọc tên nguyên liệu.

## Chuẩn bị trước khi chạy (làm ở máy)

1. Upload `data/audio_hcmc2026/` (873 file FLAC, 8,39 GB) thành Kaggle Dataset bằng `kaggle` CLI.
2. Trong notebook bật **Accelerator = GPU T4 x2** và **Internet = On**.
   Internet bắt buộc vì phải `pip install` và tải model; bật nó cần xác thực số điện thoại.
3. Sửa `DATASET_DIR` ở ô Cấu hình cho khớp slug dataset của bạn.

## Chạy thế nào

Bấm **Save Version → Save & Run All (Commit)**. Notebook chạy nền, đóng trình duyệt vẫn tiếp tục.

⚠️ **Ước tính: ~2,5–3 giờ với T4 x2** nên gần như chắc chắn xong trong một phiên.
Nhưng nếu vì lý do gì đó phải chạy lần hai, **đọc kỹ mục "Chạy tiếp" ở ô Cấu hình** —
`/kaggle/working` KHÔNG tự giữ lại giữa các lần commit.

In [ ]:
!pip install -q faster-whisper

## Cấu hình

### ⚠️ Đường dẫn dataset: dùng SLUG, không dùng tên hiển thị

Giao diện Kaggle hiện **title** (ví dụ `audio hcmc 2026 dataset`, có dấu cách), nhưng đường dẫn
thật trong máy lấy từ **slug** — phần sau dấu `/` trong `"id"` của `dataset-metadata.json`:

```
"id": "qtcqucthng/audio-hcmc-2026"   ->   /kaggle/input/audio-hcmc-2026
```

Audio được upload dưới dạng **10 file zip**, và Kaggle **tự giải nén mỗi zip thành một thư mục con**:

```
/kaggle/input/audio-hcmc-2026/
    audio_01/  L21_V001.flac ...
    audio_02/  ...
    ...
    audio_10/
```

Ô cấu hình dùng `rglob("*.flac")` nên **tự đi vào các thư mục con, không cần sửa gì**.
Nếu gõ sai slug thì nó tự dò khắp `/kaggle/input` rồi báo cho bạn biết, thay vì chết cụt.

### ⚠️ Chạy tiếp lần thứ hai — bẫy Kaggle dễ mất sạch công

Mỗi lần commit, Kaggle **dựng lại `/kaggle/working` từ số không**. Kết quả lần trước KHÔNG
tự còn đó. Muốn chạy tiếp phải tự tay:

1. Vào **Add Data → Notebook Output**, chọn version đã chạy dở của chính notebook này.
2. Nó gắn vào `/kaggle/input/<tên-notebook>/`.
3. Thêm đường dẫn đó vào `RESUME_DIRS` bên dưới.

Bỏ bước này thì lần chạy thứ hai làm lại từ đầu cả 873 file mà không báo gì.

In [ ]:
from pathlib import Path
import os, json, time, subprocess, shutil

# Lấy từ SLUG trong dataset-metadata.json ("id": "qtcqucthng/audio-hcmc-2026"),
# KHÔNG phải cái tên hiển thị trên giao diện. Xem ô markdown ở trên.
DATASET_DIR = Path("/kaggle/input/audio-hcmc-2026")

OUT_DIR = Path("/kaggle/working/asr")

# Thư mục chứa kết quả lần chạy TRƯỚC (xem ô markdown ở trên). Để trống nếu chạy lần đầu.
RESUME_DIRS = [
    # Path("/kaggle/input/kaggle-whisper-asr/asr"),
]

MODEL      = "large-v3-turbo"   # đừng hạ xuống medium: tiếng Việt chênh rõ
LANGUAGE   = "vi"              # ép cứng, KHÔNG auto-detect
BEAM_SIZE  = 5
BATCH_SIZE = 16                # giảm còn 8 nếu OOM
MAX_HOURS  = 11.0              # tự dừng trước trần 12h của Kaggle
N_EXPECTED = 873

OUT_DIR.mkdir(parents=True, exist_ok=True)

# rglob: audio nằm trong audio_01/ ... audio_10/ vì Kaggle tự giải nén mỗi zip
# thành một thư mục con. Nếu sau này đổi cách đóng gói thì vẫn tìm được.
audio = sorted(DATASET_DIR.rglob("*.flac"))

# Gõ sai slug là lỗi hay gặp nhất -> tự dò khắp /kaggle/input thay vì chỉ báo lỗi cụt.
if not audio:
    print(f"⚠ Không thấy .flac trong {DATASET_DIR} — đang dò khắp /kaggle/input ...")
    found = sorted(Path("/kaggle/input").rglob("*.flac"))
    if found:
        DATASET_DIR = found[0].parent.parent
        audio = sorted(DATASET_DIR.rglob("*.flac"))
        print(f"  -> tìm thấy, dùng {DATASET_DIR}")
    else:
        print("  các dataset đang gắn:",
              [p.name for p in Path("/kaggle/input").glob("*")])
assert audio, "Không thấy file .flac nào — kiểm tra đã Add Data đúng dataset chưa"

if len(audio) != N_EXPECTED:
    print(f"⚠ CHỈ CÓ {len(audio)}/{N_EXPECTED} file — dataset upload thiếu? "
          f"Chạy tiếp vẫn được nhưng sẽ thiếu transcript.")

# Chép kết quả cũ vào OUT_DIR để phần sau chỉ cần nhìn một chỗ duy nhất.
restored = 0
for d in RESUME_DIRS:
    for p in Path(d).glob("*.json"):
        if not (OUT_DIR / p.name).exists():
            shutil.copy2(p, OUT_DIR / p.name)
            restored += 1

print(f"{len(audio)} file audio · khôi phục {restored} transcript từ lần chạy trước")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)

## Nạp model — MỘT model riêng cho MỖI GPU

**Kaggle có cho 2 GPU thật.** Ở ô Accelerator chọn **GPU T4 x2** thì được 2 × Tesla T4 16 GB.
Nếu chọn **P100** thì chỉ có 1. Lý do các dự án trước chỉ thấy 1 GPU thường là vì hầu hết
framework mặc định chỉ dùng `cuda:0` — phải tự chia việc mới dùng được cả hai.
Ô dưới tự nhận số GPU nên chọn P100 vẫn chạy đúng, chỉ chậm hơn ~2 lần.

### Vì sao mỗi GPU một model, không dùng chung một `pipe`

Đã đọc mã nguồn `faster_whisper 1.2.1`: `BatchedInferencePipeline` có **biến trạng thái dùng chung**
là `self.last_speech_timestamp`. Hai luồng gọi cùng một đối tượng `pipe` là **đua dữ liệu**.
Với `word_timestamps=False` (mặc định ở đây) thì nó chỉ bị ghi đè bằng `0.0` nên vô hại —
**nhưng dựa vào chi tiết đó là mong manh**, đổi một tham số là hỏng ngầm mà không báo lỗi.
Mỗi GPU một model riêng thì không có gì để tranh, mà cũng chẳng tốn thêm gì
(turbo fp16 ~1,6 GB trên card 16 GB).

### `BatchedInferencePipeline` là thứ quyết định lọt hay không lọt phiên 12h

Không batching, `large-v3-turbo` chạy ~8–12x realtime ⇒ 130,7h mất 11–16 tiếng, **tràn phiên**.
Có batching ⇒ ~25–40x realtime mỗi GPU.

In [ ]:
import queue
import torch
from faster_whisper import WhisperModel, BatchedInferencePipeline

N_GPU = max(1, torch.cuda.device_count())
print(f"dùng {N_GPU} GPU")

t0 = time.perf_counter()
POOL = queue.Queue()          # mỗi luồng mượn 1 pipeline rồi trả lại
for i in range(N_GPU):
    m = WhisperModel(MODEL, device="cuda", device_index=i, compute_type="float16")
    POOL.put(BatchedInferencePipeline(model=m))
print(f"nạp {N_GPU} model xong sau {time.perf_counter() - t0:.0f}s")

## Hàm transcript một file

### 🔴 Lọc ảo giác — đo thật rồi mới chọn ngưỡng

Chạy thử trên 3 file L24 (múa lân, chỉ có trống nhạc, **không có lời**) ra kết quả bịa hoàn toàn:
`"Hãy subscribe cho kênh lalaschool"`, `"NANI!?"` — rác Whisper học từ phụ đề YouTube.

**Mọi bộ lọc thống kê tiêu chuẩn đều VÔ DỤNG ở đây**, đã đo:

| đoạn ảo giác | `no_speech_prob` | `avg_logprob` | `compression_ratio` |
|---|---|---|---|
| "Hãy subscribe cho kênh lalaschool" | **0.000** | **−0.265** | 0.91 |
| "NANI!?" | **0.000** | −0.820 | 0.43 |

Model **tự tin tuyệt đối** vào câu nó bịa ⇒ `no_speech_threshold` / `log_prob_threshold` /
`compression_ratio_threshold` không bắt được. Còn `hallucination_silence_threshold` thì
batched pipeline **hardcode `=None`** (mã nguồn dòng 435) nên cũng không dùng được.

**Tín hiệu sạch duy nhất là mật độ chữ:**

| | thấp nhất | trung vị | cao nhất |
|---|---|---|---|
| lời nói thật (125 đoạn) | **46** | 224 | 339 từ/phút |
| ảo giác (3 đoạn) | 2 | 19 | **24** từ/phút |

Khoảng trống 24 → 46 rất rộng. Chọn **`MIN_WPM = 35`** nằm giữa: giữ 125/125 lời thật,
loại 3/3 ảo giác.

Đoạn bị loại được **giữ lại trong khoá `dropped`** chứ không vứt im lặng — để còn kiểm lại
xem ngưỡng có cắt nhầm không.

### Hai tham số còn lại

- **`vad_filter=True`** — bỏ đoạn im lặng/nhạc trước khi vào model: nhanh hơn nhiều và bớt
  chỗ cho model bịa.
- **`temperature=0`** — tất định. Đã kiểm: cùng file chạy 2 lần cho WER = 0,0000.

Không cần truyền `condition_on_previous_text`: batched pipeline **tự ép `=False`** (dòng 436).

Ghi **một JSON mỗi video, xong file nào lưu ngay**. Ghi ra `.part` rồi đổi tên, để file dở dang
không bị tưởng là đã xong.

In [ ]:
MIN_WPM = 35        # xem ô markdown ở trên: lời thật >= 46, ảo giác <= 24

# Lưới thứ hai, chỉ những cụm KHÔNG THỂ xuất hiện trong chương trình HTV.
# CỐ Ý không chặn "cảm ơn các bạn đã theo dõi" hay "đăng ký kênh" — đó là lời
# kết thật của bản tin, chặn là mất nội dung thật. Bộ lọc từ/phút đã bắt được rồi.
BLOCK = ("subscribe", "lalaschool", "la la school", "ghiền mì gõ")


def is_junk(text: str, start: float, end: float):
    """Trả lý do bị loại, hoặc None nếu giữ."""
    low = text.lower()
    for b in BLOCK:
        if b in low:
            return f"blocklist:{b}"
    wpm = len(text.split()) / (max(end - start, 0.1) / 60)
    if wpm < MIN_WPM:
        return f"wpm:{wpm:.0f}"
    return None


def transcribe_one(pipe, path: Path) -> dict:
    segs, info = pipe.transcribe(
        str(path),
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        batch_size=BATCH_SIZE,
        vad_filter=True,
        temperature=0,
    )
    out, dropped, last, run = [], [], None, 0
    for s in segs:
        txt = s.text.strip()
        if not txt:
            continue
        # Lặp y hệt >3 lần liên tiếp là ảo giác, không phải lời nói.
        # Giữ 3 lần đầu để không xoá nhầm điệp khúc bài hát.
        run = run + 1 if txt == last else 0
        last = txt
        if run >= 3:
            dropped.append({"start": round(s.start, 2), "text": txt, "why": "lặp"})
            continue
        why = is_junk(txt, s.start, s.end)
        if why:
            dropped.append({"start": round(s.start, 2), "text": txt, "why": why})
            continue
        out.append({"start": round(s.start, 2), "end": round(s.end, 2), "text": txt})
    # Giữ lại phần bị loại để còn soi được, đừng vứt im lặng.
    return {"video_id": path.stem, "duration": round(info.duration, 2),
            "segments": out, "dropped": dropped}


def work(path: Path):
    dst = OUT_DIR / f"{path.stem}.json"
    if dst.exists():
        return path.stem, "bỏ qua", 0.0
    pipe = POOL.get()                      # mượn 1 GPU
    t = time.perf_counter()
    try:
        rec = transcribe_one(pipe, path)
    except Exception as err:
        return path.stem, f"LỖI: {type(err).__name__}: {err}"[:120], 0.0
    finally:
        POOL.put(pipe)                     # trả lại, kể cả khi lỗi
    tmp = dst.with_suffix(".part")
    tmp.write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
    tmp.replace(dst)
    return path.stem, "xong", time.perf_counter() - t

## Chạy thử 3 file trước

**Đừng bỏ ô này.** Nó tốn 1–2 phút và bắt được mọi lỗi cấu hình (sai đường dẫn, OOM, sai tham số)
trước khi bạn đợi 3 tiếng rồi mới phát hiện. Nhìn kỹ đoạn text in ra: phải là **tiếng Việt có nghĩa**,
không phải chuỗi lặp.

In [ ]:
_by_size = sorted(audio, key=lambda p: p.stat().st_size)
# 1 file NHỎ NHẤT (thường là L24 múa lân, không có lời -> kiểm bộ lọc ảo giác)
# + 2 file cỡ TRUNG BÌNH (có lời nói thật -> kiểm chất lượng transcript).
# Chỉ lấy file nhỏ nhất thì không biết được model đọc tiếng Việt có ra hồn không.
probe = [_by_size[0], _by_size[len(_by_size) // 2], _by_size[len(_by_size) // 2 + 1]]

for p in probe:
    vid, status, secs = work(p)
    print(f"{vid}: {status} ({secs:.0f}s)")
    if status != "xong":
        continue
    rec = json.loads((OUT_DIR / f"{vid}.json").read_text(encoding="utf-8"))
    print(f"   {rec['duration']/60:.1f} phút · {len(rec['segments'])} đoạn giữ · "
          f"{len(rec['dropped'])} đoạn loại · "
          f"nhanh gấp {rec['duration']/max(secs,1e-9):.0f}x realtime")
    for s in rec["segments"][:3]:
        print(f"   [{s['start']:7.1f}s] {s['text'][:90]}")
    for s in rec["dropped"][:3]:
        print(f"   ✂ ({s['why']}) {s['text'][:80]}")

print("\nCần thấy gì để yên tâm chạy tiếp:")
print("  · file nhỏ nhất: 0 đoạn giữ, vài đoạn ✂ -> bộ lọc ảo giác đang hoạt động")
print("  · 2 file trung bình: hàng chục đoạn giữ, tiếng Việt ĐỌC HIỂU ĐƯỢC")
print(f"  · tổng thời gian ≈ 130,7 / (Nx × {N_GPU}) giờ, N lấy từ dòng 'nhanh gấp'")

## Vòng chạy chính

Xếp **file ngắn trước**: nếu hết giờ thì số video làm xong nhiều nhất có thể — L26 có 498 video
ngắn, làm xong chúng có giá trị hơn vài file dài.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

todo = [p for p in audio if not (OUT_DIR / f"{p.stem}.json").exists()]
todo.sort(key=lambda p: p.stat().st_size)
print(f"còn {len(todo)}/{len(audio)} file phải làm")

t0 = time.perf_counter()
done = fail = 0
ex = ThreadPoolExecutor(max_workers=N_GPU)
futs = {ex.submit(work, p): p for p in todo}
try:
    for i, f in enumerate(as_completed(futs), 1):
        vid, status, secs = f.result()
        if status == "xong":
            done += 1
        elif status.startswith("LỖI"):
            fail += 1
            print(f"  {vid}: {status}", flush=True)
        if i % 25 == 0 or i == len(todo):
            el = (time.perf_counter() - t0) / 3600
            print(f"  {i}/{len(todo)} · xong={done} lỗi={fail} · "
                  f"{el:.2f}h, còn ~{el / i * (len(todo) - i):.2f}h", flush=True)
        if (time.perf_counter() - t0) / 3600 > MAX_HOURS:
            print(f"\n⏰ Chạm mốc {MAX_HOURS}h — dừng sạch để kịp lưu kết quả.")
            print("   Commit lại, NHỚ gắn Notebook Output vào RESUME_DIRS (xem ô Cấu hình).")
            break
finally:
    ex.shutdown(wait=False, cancel_futures=True)

print(f"\n{len(list(OUT_DIR.glob('*.json')))}/{len(audio)} video đã có transcript")

## Gộp lại thành một file để tải về

Giữ `start`/`end` từng đoạn, **không gộp text cả video**: bước sau phải gán đoạn lời vào keyframe
theo `pts_time`. Mất mốc thời gian thì chỉ biết "video này có nhắc tới gà nướng" chứ không biết
**khung nào** — mà bài nộp cần `(video_id, frame_idx)`.

In [ ]:
import pandas as pd

recs = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(OUT_DIR.glob("*.json"))]
rows = [(r["video_id"], s["start"], s["end"], s["text"]) for r in recs for s in r["segments"]]
df = pd.DataFrame(rows, columns=["video_id", "start", "end", "text"])
df.to_parquet("/kaggle/working/asr_hcmc2026.parquet", index=False)

print(f"{len(df):,} đoạn · {df.video_id.nunique()} video · "
      f"{df.text.str.split().str.len().sum():,} từ")

# Chế độ batch cắt audio thành khối cố định ~30s nên đoạn sẽ DÀI.
# Con số này quyết định độ mịn của bước gán lời vào keyframe ở dưới — phải nhìn.
sec = df.end - df.start
print(f"độ dài đoạn: trung vị {sec.median():.1f}s · dài nhất {sec.max():.1f}s")

print("\nSố đoạn theo chương trình:")
print(df.groupby(df.video_id.str[:3]).size().to_string())

print("\nVài đoạn mẫu:")
for r in df.sample(min(5, len(df)), random_state=0).itertuples():
    print(f"  {r.video_id} [{r.start:7.1f}s] {r.text[:110]}")

## Kiểm tra trước khi tin kết quả

Đừng tải về rồi dùng luôn. Ba dấu hiệu hỏng thường gặp:

1. **Video rất ít chữ so với thời lượng** → VAD cắt nhầm, hoặc luồng audio hỏng.
2. **Một câu lặp lại hàng trăm lần** → ảo giác lọt lưới.
3. **Chương trình nào đó trắng trơn** → thiếu file trong dataset.

In [ ]:
dur = {r["video_id"]: r["duration"] for r in recs}
g = df.groupby("video_id").agg(
    n_seg=("text", "size"),
    n_word=("text", lambda s: s.str.split().str.len().sum()))
g["phut"] = g.index.map(dur).astype(float) / 60
g["tu_moi_phut"] = (g.n_word / g.phut).round(1)

print("10 video ÍT chữ nhất trên mỗi phút (nghi VAD cắt nhầm / audio hỏng):")
print(g.nsmallest(10, "tu_moi_phut").to_string())
print(f"\ntrung vị toàn corpus: {g.tu_moi_phut.median():.0f} từ/phút")

empty = sorted(set(dur) - set(df.video_id))
print(f"\n{len(empty)} video không còn đoạn nào sau khi lọc.")
print("  (L24 múa lân chỉ có trống nhạc nên KHÔNG có lời là đúng, không phải lỗi)")
if empty:
    import collections
    print("  theo chương trình:", dict(collections.Counter(v[:3] for v in empty)))

# --- Soi phần BỊ LOẠI: ngưỡng có cắt nhầm lời thật không? ---
# .get("dropped", []): file từ lần chạy CŨ (trước khi có bộ lọc) không có khoá này.
# Thiếu dòng này thì chạy tiếp qua RESUME_DIRS sẽ chết vì KeyError.
drops = [(r["video_id"], d["why"], d["text"])
         for r in recs for d in r.get("dropped", [])]
old = sum(1 for r in recs if "dropped" not in r)
if old:
    print(f"\n⚠ {old} file ở định dạng CŨ (chưa qua bộ lọc ảo giác) — "
          f"xoá JSON của chúng rồi chạy lại nếu muốn lọc sạch")

print(f"\n{len(drops)} đoạn bị loại tổng cộng")
if drops:
    import collections
    print("  theo loại:", dict(collections.Counter(w.split(":")[0] for _, w, _ in drops)))
    print("\n  10 đoạn bị loại DÀI NHẤT — nếu thấy câu tiếng Việt tử tế ở đây")
    print("  thì MIN_WPM đang quá cao, phải hạ xuống:")
    for vid, why, txt in sorted(drops, key=lambda x: -len(x[2]))[:10]:
        print(f"    {vid} ({why}) {txt[:85]}")

# Ảo giác biểu hiện là MỘT câu ngắn lặp đi lặp lại. Đoạn dài ~30s gần như không
# bao giờ trùng nhau, nên chỉ xét đoạn ngắn và chỉ in thứ thực sự lặp.
rep = df[df.text.str.len() < 120].text.value_counts()
rep = rep[rep > 5]
if len(rep):
    print(f"\n⚠ {len(rep)} câu ngắn lặp >5 lần (ảo giác lọt lưới):")
    for txt, n in rep.head(10).items():
        print(f"  {n:5d}x  {txt[:90]}")
else:
    print("\nKhông có câu ngắn nào lặp >5 lần — sạch")

## Xong rồi làm gì

1. Tải `asr_hcmc2026.parquet` về, để cạnh `data/processed_hcmc2026/`.
2. **Gán đoạn lời vào keyframe:** với mỗi keyframe lấy các đoạn có `start <= pts_time <= end`.
   Đây là bước biến kênh audio thành kênh tìm kiếm ở **mức khung**.
3. Đổ vào trường `asr` của index `frames` trong Elasticsearch.
4. **Đo lại trên bộ 81 query.** Mốc phải vượt: `text_en` thuần CLIP = **0,4765**.

### ⚠️ Độ mịn thời gian: đoạn dài ~30 giây

Chế độ batch cắt audio thành **khối cố định ~30s**, nên đoạn trả về dài chứ không phải từng câu
(đã đo khi chạy thử: 40 đoạn cho 17,6 phút). Keyframe cách nhau ~3s ⇒ **một đoạn phủ khoảng
10 keyframe**, tức cả 10 khung đó nhận cùng một đoạn lời.

Hệ quả phải chấp nhận: ASR **thu hẹp được về cửa sổ 30 giây**, không chỉ đích danh một khung.
Với KIS thì vẫn tốt (thu từ 177.321 khung xuống ~10 là thắng lớn), nhưng **đừng trông vào ASR
để xếp hạng trong nội bộ 10 khung đó** — việc đó vẫn là của CLIP. Đây chính là lý do phải hợp
nhất hai kênh chứ không thay thế nhau.

Muốn mịn hơn thì bật `word_timestamps=True`, nhưng nó chậm hơn đáng kể **và** kích hoạt biến
dùng chung `self.last_speech_timestamp` — lúc đó việc mỗi GPU một pipeline riêng ở trên trở
thành bắt buộc chứ không còn là đề phòng.

### Cách đọc kết quả cho đúng

Query của mình tả **hình ảnh**, ASR là **lời nói** — hai thứ không luôn trùng nhau (người dẫn
đọc tên món trong khi hình quay cái chảo). Nếu ASR thật sự có tác dụng thì cải thiện phải xuất
hiện ở **đúng L26 (3/8) và L25 (4/7)**. Điểm tăng đều khắp mọi chương trình thì nhiều khả năng
là do thứ khác chứ không phải ASR.